# Task 3 — Midterm Project: Bank Marketing
Student: Smakov Danieldiaz · SEED: 3194 (last 4 digits of student ID)

In [2]:
import re
import sys
from importlib.metadata import version, PackageNotFoundError

import numpy as np
import pandas as pd

SEED = 3194

print("Python:", sys.version.split()[0])
with open("requirements.txt") as f:
    for line in f:
        name = re.split(r"[=<>!~\[; ]", line.strip())[0]
        if name and not name.startswith("#"):
            try:
                print(f"  {name:20s} {version(name)}")
            except PackageNotFoundError:
                print(f"  {name:20s} NOT INSTALLED")

train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")

print("\ntrain:", train.shape, "| test:", test.shape)
print("Только в train:", sorted(set(train.columns) - set(test.columns)))
print("Только в test :", sorted(set(test.columns) - set(train.columns)))
print("\nКолонки train:", list(train.columns))

Python: 3.12.5
  numpy                2.3.4
  scipy                1.16.3
  pandas               2.3.3
  matplotlib           3.10.7
  scikit-learn         1.7.2
  joblib               1.5.2

train: (32940, 22) | test: (8236, 20)
Только в train: ['duration', 'y']
Только в test : []

Колонки train: ['id', 'age', 'job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'y']


In [3]:
train.head()

,id,age,job,marital,education,default,housing,loan,contact,month,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,0,39,management,single,basic.9y,unknown,yes,no,telephone,may,...,3,999,0,nonexistent,1.1,93.994,-36.4,4.856,5191.0,no
1,1,32,admin.,single,university.degree,no,yes,no,cellular,aug,...,2,999,0,nonexistent,1.4,93.444,-36.1,4.963,5228.1,no
2,2,41,technician,married,high.school,no,no,yes,cellular,aug,...,1,999,0,nonexistent,1.4,93.444,-36.1,4.963,5228.1,no
3,3,39,admin.,married,professional.course,no,yes,yes,telephone,may,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,4,51,blue-collar,married,basic.9y,unknown,no,no,telephone,jun,...,1,999,0,nonexistent,1.4,94.465,-41.8,4.864,5228.1,no


## Stage 1. Framing

The bank calls clients to sell a term deposit. For each client I need to predict how likely they are to say yes, and this has to happen before the call is made.

The call centre can only call 10% of the list. So in practice the model is used to rank clients: sort them by score and call the top 10%. Because of that, accuracy and the usual 0.5 threshold don't really matter here. What matters is how many "yes" clients end up in that top 10%.

To compare models I use average precision (area under the precision-recall curve) as the main metric. The data is imbalanced, only about 11% say yes, and AP pays more attention to the top of the ranking than ROC-AUC does. I also report ROC-AUC, since the hidden test set is scored with it. At the end I check precision, recall and lift for the top 10% on my hold-out.

The train and test files don't have the same columns. test.csv has no y, which is expected, but it also has no duration. Duration is how long the call lasted, so it's only known after the call is over. It's strongly linked to the answer (a short call almost always means no), but we score clients before calling them, so we won't have it at prediction time. If I used it, the model would look great in validation but be useless in reality. This is data leakage, so I drop duration. I also drop id, because it's just a row number.

## Stage 2 · Understanding the data

In [4]:
y_all = (train["y"] == "yes").astype(int)

print(train["y"].value_counts())
print(f"\nShare of 'yes': {y_all.mean():.4f}")
print(f"Accuracy of always predicting 'no': {1 - y_all.mean():.4f}")

y
no     29229
yes     3711
Name: count, dtype: int64

Share of 'yes': 0.1127
Accuracy of always predicting 'no': 0.8873


In [5]:
print("Real NaN in train:", train.isna().sum().sum(), "| in test:", test.isna().sum().sum())

cat_cols = [c for c in train.select_dtypes("object").columns if c != "y"]
num_cols = [c for c in train.columns if c not in cat_cols + ["id", "y", "duration"]]
print("\nCategorical:", cat_cols)
print("Numeric    :", num_cols)

unknown = pd.DataFrame({
    "train_count": (train[cat_cols] == "unknown").sum(),
    "train_share": (train[cat_cols] == "unknown").mean().round(4),
    "test_count":  (test[cat_cols] == "unknown").sum(),
})
print("\n'unknown' per categorical column:")
print(unknown[unknown["train_count"] > 0])

n999 = (train["pdays"] == 999).sum()
print(f"\npdays == 999: {n999} rows ({n999 / len(train):.4f} of train)")
print("\npdays==999 vs poutcome:")
print(pd.crosstab(train["pdays"] == 999, train["poutcome"]))
print("\npdays==999 vs previous == 0:")
print(pd.crosstab(train["pdays"] == 999, train["previous"] == 0))

print("\ndefault value counts:")
print(train["default"].value_counts())

Real NaN in train: 0 | in test: 0

Categorical: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'poutcome']
Numeric    : ['age', 'campaign', 'pdays', 'previous', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']

'unknown' per categorical column:
           train_count  train_share  test_count
job                275       0.0083          55
marital             64       0.0019          16
education         1401       0.0425         329
default           6871       0.2086        1725
housing            794       0.0241         196
loan               794       0.0241         196

pdays == 999: 31694 rows (0.9622 of train)

pdays==999 vs poutcome:
poutcome  failure  nonexistent  success
pdays                                  
False         116            0     1130
True         3248        28446        0

pdays==999 vs previous == 0:
previous  False  True 
pdays                 
False      1246      0
True       

In [6]:
from scipy.stats import spearmanr

print("First 20 months by id:", train.sort_values("id")["month"].head(20).tolist())
for col in ["euribor3m", "nr.employed", "emp.var.rate"]:
    rho, _ = spearmanr(train["id"], train[col])
    print(f"Spearman(id, {col}) = {rho:+.3f}")

First 20 months by id: ['may', 'aug', 'aug', 'may', 'jun', 'aug', 'may', 'apr', 'jul', 'nov', 'nov', 'sep', 'aug', 'jul', 'aug', 'nov', 'may', 'apr', 'jun', 'jun']
Spearman(id, euribor3m) = -0.002
Spearman(id, nr.employed) = -0.002
Spearman(id, emp.var.rate) = -0.001
